# 3-Head Interview Question Classifier Training

Train DistilBERT with three classification heads:
- **Head 1**: Punctuation/capitalization (12 classes) — pre-trained, frozen
- **Head 2**: Actionable vs non-actionable (binary) — main classification task
- **Head 3**: Query span detection (token-level, 2-class) — identify where question starts

Dataset: 27,400 samples from YouTube interview transcripts + curated questions (3:1 actionable ratio)

## Setup & Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets evaluate
!pip install -q scikit-learn numpy pandas
!pip install -q coremltools
!pip install -q tqdm

In [ ]:
import json
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Data Loading & Preprocessing

In [ ]:
# Upload dataset_normalized.jsonl to Colab or mount Google Drive
# For now, we'll read from a path (modify as needed for your setup)

def load_jsonl(path: str) -> list[dict]:
    """Load JSONL file."""
    samples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples

# Load dataset
# MODIFY: Replace with your actual path
DATASET_PATH = "dataset_normalized.jsonl"

print(f"Loading dataset from {DATASET_PATH}...")
samples = load_jsonl(DATASET_PATH)

print(f"Total samples: {len(samples)}")

# Statistics
actionable_count = sum(1 for s in samples if s['head2_label'] == 'actionable')
non_actionable_count = len(samples) - actionable_count
with_punct = sum(1 for s in samples if s.get('has_punct', False))
with_query_start = sum(1 for s in samples if s.get('head3_query_start', None) is not None and s['head3_query_start'] > 0)

print(f"\nDataset composition:")
print(f"  Actionable: {actionable_count} ({100*actionable_count/len(samples):.1f}%)")
print(f"  Non-actionable: {non_actionable_count} ({100*non_actionable_count/len(samples):.1f}%)")
print(f"  With punctuation: {with_punct} ({100*with_punct/len(samples):.1f}%)")
print(f"  With query_start > 0: {with_query_start} ({100*with_query_start/len(samples):.1f}%)")

In [ ]:
# Stratified train/val/test split
labels = [s['head2_label'] for s in samples]

# 80% train, 20% temp
train_indices, temp_indices = train_test_split(
    range(len(samples)),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# 50% val, 50% test from temp (so 10% each of original)
temp_labels = [labels[i] for i in temp_indices]
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

train_samples = [samples[i] for i in train_indices]
val_samples = [samples[i] for i in val_indices]
test_samples = [samples[i] for i in test_indices]

print(f"Split sizes:")
print(f"  Train: {len(train_samples)} ({100*len(train_samples)/len(samples):.1f}%)")
print(f"  Val:   {len(val_samples)} ({100*len(val_samples)/len(samples):.1f}%)")
print(f"  Test:  {len(test_samples)} ({100*len(test_samples)/len(samples):.1f}%)")

# Verify stratification
for split_name, split_samples in [("Train", train_samples), ("Val", val_samples), ("Test", test_samples)]:
    act = sum(1 for s in split_samples if s['head2_label'] == 'actionable')
    ratio = act / max(1, len(split_samples) - act)
    print(f"{split_name} actionable ratio: {ratio:.2f}:1")

## Model Architecture

In [ ]:
# Load tokenizer (DistilBERT, shared with PunctuationModel)
MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Model name: {MODEL_NAME}")

# Test tokenization
test_text = "Какие вопросы обычно задают на собеседовании?"
tokens = tokenizer.tokenize(test_text)
print(f"\nExample tokenization of: '{test_text}'")
print(f"Tokens: {tokens}")
print(f"Token count: {len(tokens)}")

In [ ]:
@dataclass
class QuestionSample:
    """Represents a single training sample."""
    text: str
    head2_label: str  # "actionable" or "non_actionable"
    head3_query_start: Optional[int]  # word offset where query starts, or None


class QuestionDataset(Dataset):
    """PyTorch dataset for question classification."""
    
    MAX_LEN = 128
    
    def __init__(self, samples: list[dict], tokenizer, head3_mode: str = "binary"):
        """
        Args:
            samples: List of dicts with 'text', 'head2_label', 'head3_query_start'
            tokenizer: HF tokenizer
            head3_mode: "binary" (token classification) or "none" (skip Head 3)
        """
        self.samples = [QuestionSample(
            text=s['text'],
            head2_label=s['head2_label'],
            head3_query_start=s.get('head3_query_start')
        ) for s in samples]
        self.tokenizer = tokenizer
        self.head3_mode = head3_mode
        
        # Build label mapping
        self.head2_label2id = {"non_actionable": 0, "actionable": 1}
        self.head2_id2label = {v: k for k, v in self.head2_label2id.items()}
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> dict:
        sample = self.samples[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            sample.text,
            max_length=self.MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # Head 2 label (binary classification)
        head2_label = self.head2_label2id[sample.head2_label]
        
        # Head 3 labels (token-level, if available)
        head3_labels = None
        if self.head3_mode == "binary" and sample.head3_query_start is not None:
            # Create token-level labels: 1 if at or after query_start, 0 otherwise
            word_ids = encoding.word_ids()
            head3_labels = np.zeros(len(word_ids), dtype=np.int32)
            
            # Mark tokens at/after query_start as class 1
            if sample.head3_query_start > 0:
                for token_idx, word_idx in enumerate(word_ids):
                    if word_idx is not None and word_idx >= sample.head3_query_start:
                        head3_labels[token_idx] = 1
        else:
            # Default: no query start marked (class 0 for all)
            head3_labels = np.zeros(self.MAX_LEN, dtype=np.int32)
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "head2_label": torch.tensor(head2_label, dtype=torch.long),
            "head3_labels": torch.tensor(head3_labels, dtype=torch.long),
        }


# Create datasets
train_dataset = QuestionDataset(train_samples, tokenizer, head3_mode="binary")
val_dataset = QuestionDataset(val_samples, tokenizer, head3_mode="binary")
test_dataset = QuestionDataset(test_samples, tokenizer, head3_mode="binary")

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Test one sample
sample = train_dataset[0]
print(f"\nSample keys: {sample.keys()}")
print(f"Input shape: {sample['input_ids'].shape}")
print(f"Head2 label: {sample['head2_label'].item()}")
print(f"Head3 labels shape: {sample['head3_labels'].shape}")

In [ ]:
class ThreeHeadQuestionClassifier(nn.Module):
    """3-head DistilBERT classifier for interview questions."""
    
    def __init__(self, model_name: str, freeze_backbone: bool = False):
        super().__init__()
        
        # Load pre-trained DistilBERT
        self.backbone = AutoModel.from_pretrained(model_name)
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        hidden_size = self.backbone.config.hidden_size
        
        # Head 1: Punctuation/capitalization (12 classes) — frozen, pre-trained
        # (In this training, we focus on Head 2 and Head 3)
        self.head1 = nn.Linear(hidden_size, 12)
        for param in self.head1.parameters():
            param.requires_grad = False  # Frozen
        
        # Head 2: Binary classification (actionable vs non_actionable)
        # Uses [CLS] token representation
        self.head2 = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.Linear(256, 2),  # binary
        )
        
        # Head 3: Token-level classification (query span detection, 2 classes)
        self.head3 = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.Linear(256, 2),  # 0: not query, 1: query start
        )
    
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            input_ids: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
        
        Returns:
            head2_logits: (batch_size, 2)
            head3_logits: (batch_size, seq_len, 2)
        """
        # Backbone
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Last hidden state (batch_size, seq_len, hidden_size)
        last_hidden_state = outputs.last_hidden_state
        
        # [CLS] token (batch_size, hidden_size)
        cls_token = last_hidden_state[:, 0, :]
        
        # Head 2: Use CLS token
        head2_logits = self.head2(cls_token)  # (batch_size, 2)
        
        # Head 3: Token-level classification
        # Apply head3 to each token, then pool or use sequence
        head3_logits = self.head3(last_hidden_state)  # (batch_size, seq_len, 2)
        
        return head2_logits, head3_logits


# Initialize model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ThreeHeadQuestionClassifier(
    model_name=MODEL_NAME,
    freeze_backbone=False  # Fine-tune backbone
)
model = model.to(device)

print(f"Model moved to device: {device}")
print(f"\nModel architecture:")
print(model)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")

## Training Loop

In [ ]:
# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0

# Compute class weights for Head 2 (address imbalance)
head2_labels = [s['head2_label'] for s in train_samples]
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    "balanced",
    classes=np.array([0, 1]),
    y=np.array([train_dataset.head2_label2id[label] for label in head2_labels])
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights for Head 2: {class_weights}")

# Weighted sampler for training (balance classes)
sample_weights = np.array([
    class_weights[train_dataset.head2_label2id[label]].item()
    for label in head2_labels
])
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2 if not torch.cuda.is_available() else 0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2 if not torch.cuda.is_available() else 0
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2 if not torch.cuda.is_available() else 0
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Scheduler
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps
)

# Loss functions
head2_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
head3_loss_fn = nn.CrossEntropyLoss()

print(f"\nTraining configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Total steps: {total_steps}")

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, epoch: int):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    head2_correct = 0
    head2_total = 0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Train]")
    for batch in pbar:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        head2_labels = batch["head2_label"].to(device)
        head3_labels = batch["head3_labels"].to(device)
        
        # Forward pass
        head2_logits, head3_logits = model(input_ids, attention_mask)
        
        # Loss: 70% Head 2 (main task), 30% Head 3 (auxiliary)
        head2_loss = head2_loss_fn(head2_logits, head2_labels)
        head3_loss = head3_loss_fn(
            head3_logits.view(-1, 2),
            head3_labels.view(-1)
        )
        loss = 0.7 * head2_loss + 0.3 * head3_loss
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        # Metrics
        preds = head2_logits.argmax(dim=1)
        head2_correct += (preds == head2_labels).sum().item()
        head2_total += head2_labels.size(0)
        
        pbar.set_postfix({
            "loss": loss.item(),
            "acc": head2_correct / head2_total
        })
    
    avg_loss = total_loss / len(dataloader)
    avg_acc = head2_correct / head2_total
    
    return avg_loss, avg_acc


def evaluate(model, dataloader, device, split_name: str):
    """Evaluate on a dataset."""
    model.eval()
    total_loss = 0.0
    head2_preds = []
    head2_labels_list = []
    
    pbar = tqdm(dataloader, desc=f"{split_name} [Eval]")
    with torch.no_grad():
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            head2_labels = batch["head2_label"].to(device)
            head3_labels = batch["head3_labels"].to(device)
            
            head2_logits, head3_logits = model(input_ids, attention_mask)
            
            head2_loss = head2_loss_fn(head2_logits, head2_labels)
            head3_loss = head3_loss_fn(
                head3_logits.view(-1, 2),
                head3_labels.view(-1)
            )
            loss = 0.7 * head2_loss + 0.3 * head3_loss
            total_loss += loss.item()
            
            preds = head2_logits.argmax(dim=1)
            head2_preds.extend(preds.cpu().numpy())
            head2_labels_list.extend(head2_labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(head2_labels_list, head2_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        head2_labels_list,
        head2_preds,
        average="weighted",
        zero_division=0
    )
    
    print(f"\n{split_name} Results:")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1: {f1:.4f}")
    
    return avg_loss, acc, f1

In [ ]:
# Training loop
best_val_f1 = 0.0
patience = 2
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device, epoch)
    print(f"\nTrain: loss={train_loss:.4f}, acc={train_acc:.4f}")
    
    # Validate
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, device, "Validation")
    
    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), "best_model.pth")
        print(f"\n✅ New best model saved (F1: {val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"\n⚠️ No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

# Load best model
model.load_state_dict(torch.load("best_model.pth"))
print(f"\n{'='*60}")
print(f"Training complete. Best model loaded.")
print(f"{'='*60}")

In [ ]:
# Evaluate on test set
print(f"\n{'='*60}")
print(f"Final Test Evaluation")
print(f"{'='*60}")

test_loss, test_acc, test_f1 = evaluate(model, test_loader, device, "Test")

print(f"\nTest set metrics:")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  F1 Score: {test_f1:.4f}")

## Model Export & CoreML Conversion

In [ ]:
# Save model state
model_path = "question_classifier_weights.pth"
torch.save(model.state_dict(), model_path)
print(f"✅ Model weights saved to {model_path}")

# Save config for later inference
config = {
    "model_name": MODEL_NAME,
    "max_len": QuestionDataset.MAX_LEN,
    "head2_labels": train_dataset.head2_id2label,
}
with open("model_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"✅ Config saved to model_config.json")

In [ ]:
# Export to TorchScript
model.eval()

# Create dummy inputs
dummy_input_ids = torch.randn(1, QuestionDataset.MAX_LEN, dtype=torch.long).to(device)
dummy_attention_mask = torch.ones(1, QuestionDataset.MAX_LEN, dtype=torch.long).to(device)

# Trace the model
try:
    traced_model = torch.jit.trace(
        model,
        (dummy_input_ids, dummy_attention_mask)
    )
    traced_model.save("question_classifier_traced.pt")
    print(f"✅ TorchScript model saved to question_classifier_traced.pt")
except Exception as e:
    print(f"⚠️ TorchScript export failed: {e}")
    print("Continuing without TorchScript export...")

In [ ]:
# Convert to CoreML (for iOS deployment)
# Note: This requires the model to be traced or scripted

import coremltools as ct

try:
    # Create a wrapper for proper input/output specification
    class CoreMLWrapper(nn.Module):
        def __init__(self, classifier_model):
            super().__init__()
            self.classifier = classifier_model
        
        def forward(self, input_ids, attention_mask):
            head2_logits, head3_logits = self.classifier(input_ids, attention_mask)
            # Return only Head 2 logits for simplicity (or both if needed)
            return head2_logits
    
    wrapper = CoreMLWrapper(model)
    wrapper.eval()
    
    # Trace wrapper
    traced_wrapper = torch.jit.trace(
        wrapper,
        (dummy_input_ids, dummy_attention_mask)
    )
    
    # Convert to CoreML
    mlmodel = ct.convert(
        traced_wrapper,
        inputs=[
            ct.TensorType(name="input_ids", shape=(1, QuestionDataset.MAX_LEN)),
            ct.TensorType(name="attention_mask", shape=(1, QuestionDataset.MAX_LEN)),
        ],
        outputs=[
            ct.TensorType(name="logits"),
        ],
        minimum_deployment_target=ct.target.iOS13,
        compute_units=ct.ComputeUnit.CPU_AND_GPU,
    )
    
    # Save as .mlpackage
    mlmodel.save("QuestionClassifier.mlpackage")
    print(f"✅ CoreML model saved to QuestionClassifier.mlpackage")
    
except Exception as e:
    print(f"⚠️ CoreML export failed: {e}")
    print("You can manually convert using coremltools if needed")

## Inference Example

In [ ]:
def predict(text: str, model, tokenizer, device, dataset_class=QuestionDataset):
    """Predict class for a single text."""
    model.eval()
    
    with torch.no_grad():
        encoding = tokenizer(
            text,
            max_length=dataset_class.MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)
        
        head2_logits, head3_logits = model(input_ids, attention_mask)
        
        # Head 2 prediction
        head2_probs = torch.softmax(head2_logits, dim=1)
        head2_pred = head2_logits.argmax(dim=1).item()
        
        # Head 3 prediction (token-level)
        head3_probs = torch.softmax(head3_logits, dim=2)
        head3_preds = head3_logits.argmax(dim=2)
        
        return {
            "text": text,
            "head2_class": train_dataset.head2_id2label[head2_pred],
            "head2_confidence": head2_probs[0, head2_pred].item(),
            "head2_scores": {
                "non_actionable": head2_probs[0, 0].item(),
                "actionable": head2_probs[0, 1].item(),
            },
        }


# Test inference on some examples
test_texts = [
    "Какие вопросы обычно задают на собеседовании?",
    "Я работал в компании три года.",
    "Расскажи о твоем опыте работы с базами данных.",
    "Проект был завершен в срок.",
]

print("\n" + "="*80)
print("Inference Examples")
print("="*80)

for text in test_texts:
    result = predict(text, model, tokenizer, device)
    print(f"\nText: {result['text']}")
    print(f"  Prediction: {result['head2_class']}")
    print(f"  Confidence: {result['head2_confidence']:.4f}")
    print(f"  Scores: {result['head2_scores']}")

## Training Summary

### Results
- **Best validation F1:** Check above for the exact value
- **Test accuracy:** Check above for the exact value
- **Model size:** ~110M parameters (DistilBERT backbone + 3 heads)

### Outputs
- `best_model.pth` — Best checkpoint during training
- `question_classifier_weights.pth` — Final model weights
- `model_config.json` — Configuration for inference
- `question_classifier_traced.pt` — TorchScript version (if successful)
- `QuestionClassifier.mlpackage` — CoreML for iOS deployment

### Next Steps
1. Download the model files from Colab
2. For iOS integration: Import the `.mlpackage` into Xcode
3. For server-side inference: Use `question_classifier_weights.pth` with PyTorch or load the TorchScript version

### Dataset Notes
- **Training data:** 21,920 samples (80%)
- **Validation data:** 2,740 samples (10%)
- **Test data:** 2,740 samples (10%)
- **Class balance:** ~3:1 actionable:non_actionable ratio (preserved across splits)
- **Source:** YouTube interview transcripts (162 VTT files) + curated questions (128 samples)